# 실습 7 · 추적 + 평가 하네스
**멀티 에이전트 시스템 구축 · 5회** — 노대일 (동의대학교 인공지능학과)

실습 3의 리드 + 워커 시스템에 **추적 기록**과 **평가**를 붙여, 에이전트 하나와 공정하게 비교합니다.
1. 모든 LLM 호출 · 도구 실행을 span 으로 기록 (JSONL)
2. 정답 근거 논문이 정해진 질문 8개로 평가 세트 구성
3. 코드 검사(인용 존재 · 실제로 읽었나 · 정답 논문 재현율) + LLM 심사(기준표 5항목)
4. 통과율 · 비용 · 시간 비교, 신뢰구간
5. 점수가 낮은 실행의 추적을 열어 실패 꼬리표 붙이기

예상 비용: 기본 설정(질문 8 × 시스템 2 × 1회)으로 수십 센트 안팎

In [ ]:
!pip -q install anthropic

In [ ]:
import os, json, time, asyncio, re, uuid, urllib.request, contextvars
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    pass
import anthropic
aclient = anthropic.AsyncAnthropic(max_retries=4)

LEAD_MODEL, WORKER_MODEL, JUDGE_MODEL = 'claude-sonnet-5-5', 'claude-haiku-5-5', 'claude-sonnet-5-5'
PRICE = {'claude-haiku-5-5': (0.10, 0.50), 'claude-sonnet-5-5': (2.0, 10.0), 'claude-opus-5-5': (4.0, 20.0)}  # $/MTok, 2026.11
LIMIT = asyncio.Semaphore(4)

DATA = 'https://raw.githubusercontent.com/Daeil-Noh/Lectures/main/multi-agent/labs/data/'
with urllib.request.urlopen(DATA + 'papers.json') as f:
    PAPERS = {p['id']: p for p in json.load(f)['papers']}

## 1. 추적기
span 하나 = LLM 호출 하나 또는 도구 실행 하나. `parent` 로 나무 모양을 만듭니다.

In [ ]:
TRACE = []
def span_start(trace, name, parent, **kw):
    s = {'trace': trace, 'span': f'{name}#{uuid.uuid4().hex[:4]}', 'name': name, 'parent': parent, 'start': time.time(), **kw}
    TRACE.append(s)
    return s

def span_end(s, **kw):
    s['end'] = time.time(); s.update(kw)
    if 'in' in s:
        pin, pout = PRICE.get(s.get('model'), (0, 0))
        s['usd'] = (s['in'] * pin + s['out'] * pout) / 1e6

def text_of(resp):
    return ''.join(b.text for b in resp.content if b.type == 'text')

## 2. 도구와 에이전트 루프 (실습 3 + 추적)

In [ ]:
def search_papers(query: str):
    words = query.split()
    scored = sorted(((sum((p['title'] + p['abstract']).count(w) for w in words), p) for p in PAPERS.values()), key=lambda x: -x[0])
    return [{'id': p['id'], 'title': p['title'], 'year': p['year'], 'area': p['area']} for s, p in scored if s][:5] \
        or {'result': '검색 결과 없음. 더 짧은 키워드로 다시 찾아 보세요.'}

def read_paper_factory(read_log):
    def read_paper(paper_id: str):
        p = PAPERS.get(paper_id.strip().upper())
        if not p:
            raise KeyError(f'{paper_id} 없음. 있는 id: P01~P12')
        read_log.add(p['id'])
        return p
    return read_paper

WORKER_TOOLS = [
    {'name': 'search_papers', 'description': '연구실 문헌 모음(가상 논문 12편)에서 키워드로 찾아 최대 5편의 id·제목·연도·분야를 돌려준다. 내용은 read_paper 로 읽는다.',
     'input_schema': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}},
    {'name': 'read_paper', 'description': '논문 id(예: P07)로 초록 전체를 읽는다. 주장을 쓰기 전에 반드시 읽는다.',
     'input_schema': {'type': 'object', 'properties': {'paper_id': {'type': 'string'}}, 'required': ['paper_id']}},
]

async def acall_tool(impl, name, args, trace, parent):
    s = span_start(trace, 'tool:' + name, parent, input=args)
    try:
        out = impl[name](**args)
        if asyncio.iscoroutine(out):
            out = await out
        span_end(s, ok=True)
        return {'content': out if isinstance(out, str) else json.dumps(out, ensure_ascii=False)[:4000]}
    except Exception as e:
        span_end(s, ok=False, error=f'{type(e).__name__}: {e}')
        return {'content': f'{type(e).__name__}: {e}', 'is_error': True}

CURRENT_AGENT = contextvars.ContextVar('current_agent', default=None)   # 지금 실행 중인 에이전트 span (워커의 부모를 찾는 데 씀)

async def arun_agent(system, task, tools, impl, model, trace, name, parent=None, max_turns=8):
    agent_span = span_start(trace, name, parent, model=model, kind='agent')
    CURRENT_AGENT.set(agent_span['span'])
    messages = [{'role': 'user', 'content': task}]
    answer = '[중단] 최대 반복 횟수 초과'
    for turn in range(max_turns):
        async with LIMIT:
            s = span_start(trace, f'{name}:llm', agent_span['span'], model=model)
            resp = await aclient.messages.create(model=model, max_tokens=2000, system=system, tools=tools, messages=messages)
            span_end(s, **{'in': resp.usage.input_tokens, 'out': resp.usage.output_tokens, 'stop': resp.stop_reason})
        messages.append({'role': 'assistant', 'content': resp.content})
        if resp.stop_reason != 'tool_use':
            answer = text_of(resp); break
        calls = [b for b in resp.content if b.type == 'tool_use']
        outs = await asyncio.gather(*[acall_tool(impl, b.name, b.input, trace, agent_span['span']) for b in calls])
        messages.append({'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': b.id, **o} for b, o in zip(calls, outs)]})
    span_end(agent_span)
    return answer

## 3. 비교할 두 시스템
같은 도구, 같은 리드 모델. 차이는 **구조**뿐입니다.

In [ ]:
WORKER_SYSTEM = ('너는 문헌 조사 워커다. 브리프의 목표와 경계 안에서만 조사한다. search_papers 로 찾고 쓸 내용은 read_paper 로 읽는다(최대 4편). '
                 '모든 주장 끝에 [P07]처럼 id를 단다. 출력 형식을 따르고 찾지 못한 것도 보고한다.')
LEAD_SYSTEM = ('너는 연구 조사 리드다. 직접 검색하지 않고 delegate 로 워커에게 맡긴다. 문헌은 12편뿐이다: 단순 질문은 워커 1명, 비교 질문은 2~3명. '
               '독립적인 조사는 한 응답에서 동시에 맡긴다. 보고를 종합할 때 [P..] id를 유지한다. 한국어로 500자 이내로 답한다.')
SINGLE_SYSTEM = ('너는 연구 조사 조교다. search_papers 로 찾고 쓸 내용은 read_paper 로 읽고 확인한다. '
                 '모든 주장 끝에 [P07]처럼 id를 단다. 찾지 못한 것은 찾지 못했다고 쓴다. 한국어로 500자 이내로 답한다.')

DELEGATE = {'name': 'delegate', 'description': '하위 조사 하나를 워커에게 맡긴다. 워커는 대화를 모르므로 필요한 맥락을 모두 쓴다.',
            'input_schema': {'type': 'object', 'properties': {'objective': {'type': 'string'}, 'output_format': {'type': 'string'},
                                                              'boundaries': {'type': 'string'}}, 'required': ['objective', 'output_format', 'boundaries']}}

async def run_multi(question, trace):
    read_log = set()
    async def delegate(objective, output_format, boundaries):
        brief = f'목표: {objective}\n출력 형식: {output_format}\n하지 말 것: {boundaries}'
        impl = {'search_papers': search_papers, 'read_paper': read_paper_factory(read_log)}
        return await arun_agent(WORKER_SYSTEM, brief, WORKER_TOOLS, impl, WORKER_MODEL, trace, 'worker',
                                parent=CURRENT_AGENT.get(), max_turns=6)
    answer = await arun_agent(LEAD_SYSTEM, question, [DELEGATE], {'delegate': delegate}, LEAD_MODEL, trace, 'lead', max_turns=6)
    return answer, read_log

async def run_single(question, trace):
    read_log = set()
    impl = {'search_papers': search_papers, 'read_paper': read_paper_factory(read_log)}
    answer = await arun_agent(SINGLE_SYSTEM, question, WORKER_TOOLS, impl, LEAD_MODEL, trace, 'single', max_turns=12)
    return answer, read_log

## 4. 평가 세트
질문마다 **정답 근거 논문**(gold)을 정해 두었습니다. 개발 중에 프롬프트를 고친 질문과 최종 비교용 질문은 원래 분리해야 하지만, 실습에서는 8개를 그대로 씁니다.

In [ ]:
EVAL = [
    ('고전 머신러닝으로 진동·센서 기반 고장 진단이나 예측을 한 논문을 찾아 필요한 데이터 양과 성능을 정리해 줘.', {'P02', 'P05', 'P09'}),
    ('우리 문헌에 양자 머신러닝이 고전 방법보다 확실히 나았다는 근거가 있어?', {'P06', 'P10', 'P12'}),
    ('네트워크 없이 설비 옆 현장 장치에서 동작하는 예지보전 방법은 어떤 게 있어?', {'P11', 'P02'}),
    ('고장 사례가 아주 적을 때 어떤 접근이 권장돼?', {'P03'}),
    ('LLM 에이전트를 제조 공정에 쓴 사례와 그때 생긴 문제는?', {'P08'}),
    ('딥러닝 이상탐지 방법들이 요구하는 데이터 양을 비교해 줘.', {'P04', 'P07', 'P11'}),
    ('실제 양자 하드웨어에서는 성능이 어떻게 달라졌어?', {'P10'}),
    ('외관 검사에서 학습 이미지가 적을 때 쓴 방법과 그 한계는?', {'P01'}),
]

## 5. 채점: 코드 검사 + LLM 심사
- 코드: 인용 id 가 존재하는가 · 그 시스템이 **실제로 읽은** 논문인가 · 정답 논문을 얼마나 인용했나(재현율)
- LLM 심사: 기준표 5항목을 **호출 한 번**에, 0~1 점수와 통과 여부. 심사자에게는 관련 초록을 근거로 준다.

In [ ]:
JUDGE = {'name': 'grade', 'description': '답변을 기준표로 채점한다.',
         'input_schema': {'type': 'object', 'properties': {
             'factual': {'type': 'number', 'description': '0~1 초록과 사실이 맞는가'},
             'citation': {'type': 'number', 'description': '0~1 인용 id가 그 주장의 근거인가'},
             'completeness': {'type': 'number', 'description': '0~1 질문에 다 답했나'},
             'source': {'type': 'number', 'description': '0~1 적절한 논문을 근거로 썼나'},
             'efficiency': {'type': 'number', 'description': '0~1 군더더기 없이 답했나'},
             'score': {'type': 'number', 'description': '0~1 종합'},
             'pass': {'type': 'boolean'},
             'evidence': {'type': 'string', 'description': '판단 근거 한두 문장'}},
             'required': ['factual', 'citation', 'completeness', 'source', 'efficiency', 'score', 'pass', 'evidence']}}
JUDGE_SYSTEM = '너는 엄격한 연구 조교 평가자다. 주어진 초록만을 사실의 기준으로 삼는다. 답이 길다고 점수를 더 주지 않는다.'

async def judge(question, answer, gold, cited, trace):
    ref_ids = sorted((gold | set(cited)) & set(PAPERS))
    refs = '\n'.join(f"[{i}] {PAPERS[i]['title']}: {PAPERS[i]['abstract']}" for i in ref_ids)
    user = f'질문: {question}\n\n참고 초록:\n{refs}\n\n평가할 답:\n{answer}'
    async with LIMIT:
        s = span_start(trace, 'judge:llm', None, model=JUDGE_MODEL)
        r = await aclient.messages.create(model=JUDGE_MODEL, max_tokens=600, system=JUDGE_SYSTEM, tools=[JUDGE],
                                          tool_choice={'type': 'tool', 'name': 'grade'}, messages=[{'role': 'user', 'content': user}])
        span_end(s, **{'in': r.usage.input_tokens, 'out': r.usage.output_tokens})
    return next(b.input for b in r.content if b.type == 'tool_use')

def code_checks(answer, gold, read_log):
    cited = sorted(set(re.findall(r'P\d{2}', answer)))
    return {'cited': cited,
            'exists': all(c in PAPERS for c in cited) if cited else False,
            'all_read': all(c in read_log for c in cited) if cited else False,
            'gold_recall': round(len(set(cited) & gold) / len(gold), 2)}

## 6. 실행
질문 하나에 대해 두 시스템을 돌리고 채점합니다. `N_REPEAT` 를 늘리면 더 믿을 만하지만 비용도 비례해서 늘어납니다.

In [ ]:
N_REPEAT = 1
RESULTS = []

async def evaluate_one(qi, question, gold, system_name, runner, rep):
    trace = f'{system_name}-q{qi}-r{rep}'
    t0 = time.time()
    answer, read_log = await runner(question, trace)
    sec = time.time() - t0
    chk = code_checks(answer, gold, read_log)
    g = await judge(question, answer, gold, chk['cited'], trace)
    usd = sum(s.get('usd', 0) for s in TRACE if s['trace'] == trace and not s['name'].startswith('judge'))
    RESULTS.append({'system': system_name, 'q': qi, 'rep': rep, 'trace': trace, 'sec': round(sec, 1), 'usd': round(usd, 4),
                    **{k: chk[k] for k in ('exists', 'all_read', 'gold_recall')}, 'judge': g['score'], 'pass': bool(g['pass']) and chk['all_read'],
                    'evidence': g['evidence'], 'answer': answer})
    print(f'{trace}: judge={g["score"]} recall={chk["gold_recall"]} ${usd:.4f} {sec:.1f}s')

for rep in range(N_REPEAT):
    for qi, (question, gold) in enumerate(EVAL):
        await asyncio.gather(evaluate_one(qi, question, gold, 'multi', run_multi, rep),
                             evaluate_one(qi, question, gold, 'single', run_single, rep))

## 7. 결과 비교와 신뢰구간
`pass` 는 심사자의 통과 판정 **그리고** 인용한 논문을 모두 실제로 읽었을 때만 참입니다.

In [ ]:
import pandas as pd, math
res = pd.DataFrame(RESULTS)
summary = res.groupby('system').agg(n=('pass', 'size'), pass_rate=('pass', 'mean'), judge=('judge', 'mean'),
                                    gold_recall=('gold_recall', 'mean'), usd=('usd', 'sum'), sec=('sec', 'mean')).round(3)
def ci(p, n):
    h = 1.96 * math.sqrt(p * (1 - p) / n) if n else 0
    return f'{max(0, p - h):.2f}~{min(1, p + h):.2f}'
summary['pass_95ci'] = [ci(r.pass_rate, r.n) for r in summary.itertuples()]
summary

In [ ]:
# 질문별로 나란히 (짝지은 비교)
paired = res.pivot_table(index='q', columns='system', values='judge', aggfunc='mean').round(2)
paired['차이(multi-single)'] = (paired.get('multi', 0) - paired.get('single', 0)).round(2)
paired

## 8. 추적 열어 보기
추적은 JSONL 로 저장합니다(한 줄에 span 하나). 점수가 가장 낮은 실행의 타임라인을 그립니다.

In [ ]:
with open('traces.jsonl', 'w', encoding='utf-8') as f:
    for s in TRACE:
        f.write(json.dumps({k: v for k, v in s.items() if k != 'input' or len(str(v)) < 200}, ensure_ascii=False, default=str) + '\n')
print('저장: traces.jsonl', len(TRACE), 'spans')

worst = res.sort_values('judge').iloc[0]
print('가장 낮은 점수:', worst['trace'], worst['judge'], '|', worst['evidence'])
spans = [s for s in TRACE if s['trace'] == worst['trace'] and 'end' in s]
t0 = min(s['start'] for s in spans)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 0.35 * len(spans) + 1))
for i, s in enumerate(spans):
    color = '#b3261e' if s.get('ok') is False else ('#0b6e63' if s['name'].startswith(('lead', 'single')) else '#c2571a' if s['name'].startswith('worker') else '#8a93a6')
    ax.barh(i, s['end'] - s['start'], left=s['start'] - t0, color=color)
    label = s['name'] + (' ' + str(s.get('input', ''))[:24] if s['name'].startswith('tool') else '')
    ax.text(s['start'] - t0, i, ' ' + label, va='center', fontsize=7)
ax.set_yticks([]); ax.set_xlabel('seconds'); ax.set_title(worst['trace'])
plt.tight_layout(); plt.show()

In [ ]:
# 직접 해 보기: 점수가 낮은 실행 세 개를 골라 추적과 답을 읽고 실패 꼬리표를 붙이세요.
# 갈래: '명세·설계' / '에이전트 간 어긋남' / '검증·종료'. 고칠 곳도 한 줄씩.
low = res.sort_values('judge').head(3)[['trace', 'judge', 'gold_recall', 'all_read', 'evidence']]
TAGS = {t: {'갈래': '?', '증상': '?', '고칠 곳': '?'} for t in low['trace']}
print(low.to_string())
TAGS

## 9. 비용 손잡이 맛보기: 프롬프트 캐싱
매번 같은 긴 앞부분(여기서는 문헌 12편 전체를 지침에 넣은 경우)에 `cache_control` 을 달면, 두 번째 호출부터 `cache_read_input_tokens` 로 싸게 읽습니다.
캐시는 일정 길이 이상의 앞부분에서만 동작하고 몇 분 뒤 만료됩니다(가격·조건은 Claude 문서 확인).

In [ ]:
import anthropic as _a
client = _a.Anthropic(max_retries=4)
corpus = '\n'.join(f"[{p['id']}] {p['title']} ({p['year']}): {p['abstract']}" for p in PAPERS.values())
system_blocks = [{'type': 'text', 'text': '너는 연구 조교다. 아래 문헌만 근거로 답하고 [P..] id를 단다.\n\n' + corpus,
                  'cache_control': {'type': 'ephemeral'}}]
for q in ['양자 계열 논문들의 공통 한계는?', '데이터가 적을 때 쓸 만한 방법은?']:
    r = client.messages.create(model=LEAD_MODEL, max_tokens=300, system=system_blocks, messages=[{'role': 'user', 'content': q}])
    u = r.usage
    print(f"입력 {u.input_tokens} · 캐시 쓰기 {getattr(u, 'cache_creation_input_tokens', 0)} · 캐시 읽기 {getattr(u, 'cache_read_input_tokens', 0)}")

### 대량 평가는 일괄 처리(Message Batches)로
기다려도 되는 평가(질문 수백 개 × 반복)는 Message Batches API로 보내면 모든 토큰이 50% 할인됩니다. 모양만 보면:
```python
batch = client.messages.batches.create(requests=[
    {'custom_id': f'q{i}', 'params': {'model': JUDGE_MODEL, 'max_tokens': 600, 'messages': [...]}}
    for i, ... in enumerate(...)])
# 나중에: client.messages.batches.retrieve(batch.id) → 끝나면 results 로 받기
```

## 정리
- 추적이 있어야 **어디서 왜** 실패했는지 말할 수 있다.
- 평가는 코드 검사 + LLM 심사 + 사람 표본, 그리고 **같은 예산의 기준선**과 신뢰구간.
- 질문 8개의 결과로는 큰 차이만 주장할 수 있다. 미니 프로젝트 보고서에 이 한계를 적는다.

**AI에게 물어볼 만한 질문**: "이 summary 표와 paired 표를 보고, 멀티 에이전트가 더 낫다고 주장하려면 무엇을 더 실험해야 하는지 설계를 제안해 줘."